# Interactive Denoising Lab — PandaOmron — jitter-fix checkpoints (v4.5)

v4's measurements (seed spread, solver convergence, residual roughness) on the checkpoints
that answer the open questions about `apply_jitter_fix`. Both observations are the overfit
runs' training state (`ep000_step010`) and the state two chunks later (`ep000_step012`).

| label | checkpoint | question |
|---|---|---|
| BASE | no LoRA | the floor (leak gain ≈ 0.09 in v4) |
| `old-target it17` | old `overfit_step10_v5` resume, `iter_0017` (success 0.73) | old target `a − ε`; the leak reference (gain 1.0) |
| `fix p0.125 it05` | jitter-fix run, `iter_0005` | A/B: two updates from `iter_0003` at λ = 0.125 … |
| `fix p0.25 it05` | `…_res3_plam0.25`, `iter_0005` | … against λ = 0.25, same parent |
| `fix p0.25 it06` | `…_res3_plam0.25`, `iter_0006` | success 0.92 with the sampler cap at 0.75 |
| `fix p0.25 cap0.9 it08` | `…_res7_BMPARD0.9`, `iter_0008` | success 0.92 after the cap change |
| `fix p0.25 cap0.9 it09` | `…_res7_BMPARD0.9`, `iter_0009` | latest, after the erosion spike and drift jump |

The headline is the **leak table** after the seed-sweep summaries: how strongly each seed's
initial noise shows up in its actions, relative to the old target. v4 measured the λ = 0.125
run's `iter_0003` / `0005` / `0006` at 0.15 / 0.17 / 0.18.

How it works is unchanged from v4: one model load; LoRA injected once (rank 16, alpha 32,
not merged) and swapped per checkpoint; backbone features encoded once per observation; BASE
runs with the adapters disabled. `CHECKPOINTS` is `{label: path}` because two checkpoints
share the directory name `iter_0005`.


In [ ]:
# Cell 1: Imports + config
import sys, os, json
import numpy as np
import torch
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from scripts.denoising_lab.denoising_lab import DenoisingLab, TrajectoryVisualizer

MODEL_PATH = "nvidia/GR00T-N1.6-3B"
EMBODIMENT_TAG = "robocasa_panda_omron"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Observation .npz captured by interactive_rollout.py. Change this (or just call
# analyze_all_checkpoints("...") with another path) for any other observation.
OBS_PATH = "/home/ubuntu/results/npz_save_CoffeeServeMug_v2/ep000_step010.npz"

EEF_KEY = "end_effector_position"
N_SEEDS = 100              # seeds per checkpoint (matches v1 Cell 11)

# LoRA config — MUST match what the checkpoints were trained with.
# (GRPOConfig defaults; every checkpoint below was trained with them.)
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

# Also sweep the BASE model (no LoRA) by disabling the injected adapter. Produces the
# same composite figure and an entry in the cross-checkpoint summary. See Cell 4.
INCLUDE_BASE_MODEL = True

# Every figure's data is also saved to PLOT_DATA_DIR/<name>.npz so the plots can be
# re-drawn (e.g. smoothed) offline without the GPU. File list: "Saved plot data" below.
PLOT_DATA_DIR = os.path.join(REPO_ROOT, "scripts", "denoising_lab", "notebooks", "plot_data_v4.5")
os.makedirs(PLOT_DATA_DIR, exist_ok=True)


def save_plot_data(name, meta, **arrays):
    """Write arrays + a JSON meta string to PLOT_DATA_DIR/<name>.npz (loads without pickle)."""
    path = os.path.join(PLOT_DATA_DIR, f"{name}.npz")
    np.savez(path, meta_json=json.dumps(meta), **{k: np.asarray(v) for k, v in arrays.items()})
    print(f"Plot data saved: {path}")

# Checkpoints: {label: dir relative to REPO_ROOT holding lora_weights.pt}. Labels name
# every figure, table and plot-data row; see the table in the intro for the roles.
CHECKPOINTS = {
    "old-target it17": "grpo_data/overfit_step10/overfit_step10_v5_lr5.9e-5_PAWS_plam0.25_nlam0.05_loclip0.08_hiclip0.2_kll0.1_klb0.1_tratio1.75_mbrenorm_BMPARD_nojitterpair_1epoch_gs12_ng4_mb64_resumeiter11_tratio2.25_IAG0.15/iter_0017/",
    "fix p0.125 it05": "grpo_data/overfit_step10/overfit_step10_v6_lr5e-5_plam0.125_nlam0_nojitterpair_jitterfix_loclip0.005_rel1.0_hiclip0.2_kll0.1_klb0.1_noPAWS_iterrenorm_BMPARD_3epoch_gs12_ng4_mb128_IAG0.3_postreopen2/iter_0005/",
    "fix p0.25 it05": "grpo_data/overfit_step10/overfit_step10_v6_lr5e-5_plam0.125_nlam0_nojitterpair_jitterfix_loclip0.005_rel1.0_hiclip0.2_kll0.1_klb0.1_noPAWS_iterrenorm_BMPARD_3epoch_gs12_ng4_mb128_IAG0.3_postreopen2_res3_plam0.25/iter_0005/",
    "fix p0.25 it06": "grpo_data/overfit_step10/overfit_step10_v6_lr5e-5_plam0.125_nlam0_nojitterpair_jitterfix_loclip0.005_rel1.0_hiclip0.2_kll0.1_klb0.1_noPAWS_iterrenorm_BMPARD_3epoch_gs12_ng4_mb128_IAG0.3_postreopen2_res3_plam0.25/iter_0006/",
    "fix p0.25 cap0.9 it08": "grpo_data/overfit_step10/overfit_step10_v6_lr5e-5_plam0.125_nlam0_nojitterpair_jitterfix_loclip0.005_rel1.0_hiclip0.2_kll0.1_klb0.1_noPAWS_iterrenorm_BMPARD_3epoch_gs12_ng4_mb128_IAG0.3_postreopen2_res3_plam0.25_res7_BMPARD0.9/iter_0008/",
    "fix p0.25 cap0.9 it09": "grpo_data/overfit_step10/overfit_step10_v6_lr5e-5_plam0.125_nlam0_nojitterpair_jitterfix_loclip0.005_rel1.0_hiclip0.2_kll0.1_klb0.1_noPAWS_iterrenorm_BMPARD_3epoch_gs12_ng4_mb128_IAG0.3_postreopen2_res3_plam0.25_res7_BMPARD0.9/iter_0009/",
}
# Leak-table reference: gain 1.0 = this checkpoint's noise pass-through.
LEAK_REF = "old-target it17"

n_runs = len(CHECKPOINTS) + (1 if INCLUDE_BASE_MODEL else 0)
print(f"Device: {DEVICE}")
print(f"{len(CHECKPOINTS)} checkpoints"
      f"{' + base model' if INCLUDE_BASE_MODEL else ''} = {n_runs} runs, {N_SEEDS} seeds each")
print(f"Plot data -> {PLOT_DATA_DIR}")

### Saved plot data

Every figure's data is written to `PLOT_DATA_DIR` (`scripts/denoising_lab/notebooks/plot_data_v4.5/`),
one `.npz` per figure family and observation, so the plots can be re-drawn (e.g. smoothed)
offline without the GPU. `<obs>` is the observation file stem, e.g. `ep000_step010`.

| file | figures |
|---|---|
| `seed_spread__<obs>.npz` | composite seed-spread figure per run (3D panel, X/Y/Z traces, mean±std) and the final-std summary bar chart |
| `solver_convergence__<obs>.npz` | solver-convergence figure: pathjerk / rel_disp / drift vs num_steps |
| `roughness__<obs>.npz` | that observation's row of the roughness figure: HF and M vs tau, roughness bars, spectrum |

Each file holds a `runs` array (run labels in plot order) and a `meta_json` string with the
observation path, checkpoint paths and array shapes. Camera views are not saved; they come
straight from the observation `.npz`. `seed_spread` files also hold `act_norm` and `noise`
(run, seed, step, xyz): each seed's raw normalized EEF action and its initial noise. The
leak table is computed from them and has no file of its own.

```python
d = np.load(os.path.join(PLOT_DATA_DIR, "seed_spread__ep000_step010.npz"))
meta = json.loads(d["meta_json"].item())
d["all_pos"].shape   # (run, seed, timestep, xyz)
```


In [ ]:
# Cell 2: Load model (once)
lab = DenoisingLab(MODEL_PATH, EMBODIMENT_TAG, device=DEVICE)
print(f"Model loaded. Action horizon: {lab.action_horizon}, Action dim: {lab.action_dim}")
print(f"Inference timesteps: {lab.num_inference_timesteps}, Timestep buckets: {lab.num_timestep_buckets}")

In [ ]:
# Cell 2.5: Inject LoRA ONCE into the DiT — do NOT load a checkpoint yet, do NOT merge.
#
# We inject the adapter shell here and reuse it for every checkpoint: the loop below
# calls load_lora_checkpoint(...) to swap each checkpoint's A/B matrices into these
# same modules. We deliberately skip merge_lora_weights() — merging is irreversible
# and would bake one checkpoint into the base weights, making it impossible to cleanly
# load the next one OR to disable the adapter for the base-model run. The cost is a
# small per-forward LoRA matmul (a few extra seconds per 100-seed sweep), which is fine.
import sys as _sys
_grpo_dir = os.path.join(REPO_ROOT, "scripts", "grpo")
if _grpo_dir not in _sys.path:
    _sys.path.insert(0, _grpo_dir)

# disabled_adapters: context manager that zeros the LoRA contribution for the base-model
# sweep — the DiT then produces its pretrained velocity field, no second copy in VRAM.
from lora_dit import (
    apply_lora_to_dit,
    load_lora_checkpoint,
    disabled_adapters,
    print_trainable_params,
)

apply_lora_to_dit(lab.model, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)

# Pin the freshly-injected LoRA Linears to the DiT's device/dtype. Done ONCE:
# load_lora_checkpoint copies weights in place via load_state_dict, so loaded adapters
# inherit this device/dtype on every subsequent load (no re-.to() needed).
lab.model.action_head.model.to(device=lab.device, dtype=lab.dtype)
lab.model.eval()

print_trainable_params(lab.model)  # ~14.5M LoRA params, rest frozen

## Per-checkpoint seed sweep

`run_seed_sweep` denoises 100 noise seeds per checkpoint (seed `s` starts every checkpoint from
the same initial noise), decodes each to an EEF path, prints the spread and jerk stats, and draws
one composite figure (3D panel, X/Y/Z traces, mean ± std). Jerk is the 3rd difference of the
path. Each jerk line is taken across seeds of one number per seed: mean `abs(jerk)` over the 14
windows, `abs(jerk)` at the last window, and the scale-free `sum abs(jerk) / path length` (its
`3D` row is `pathjerk`).


In [ ]:
# Cell 3: run_seed_sweep — reproduces v1 Cell 11 ("Compare seeds") for one
# (features, checkpoint) pair. Sweeps n_seeds noise seeds through the 4-step denoiser,
# decodes each to an EEF trajectory, prints per-component spread stats and jerk /
# smoothness stats, then draws ONE composite figure: a large 3D EEF-trajectory panel
# (left), the X/Y/Z component traces (top-right), and the x/y/z mean+-std spread
# (bottom-right). Returns a stats dict for the cross-checkpoint summary.
def _seed_stats_line(v, l):
    """Print mean/var/std/range across seeds of a per-seed quantity v, shape (n_seeds,)."""
    print(f"  {l}: mean={v.mean():.4g}, \tvar={v.var():.4g}, \tstd={v.std():.4g}, \trange={np.ptp(v):.4g}")


def _action_slice(key_suffix):
    """Columns of one action key in the raw (padded) model output, from the checkpoint layout."""
    tag = lab.embodiment_tag.value
    norm = lab.processor.state_action_processor.norm_params[tag]["action"]
    start = 0
    for k in lab.modality_configs["action"].modality_keys:
        d = norm[k]["dim"]
        d = int(d.item() if hasattr(d, "item") else d)
        if k.split(".")[-1] == key_suffix:
            return slice(start, start + d)
        start += d
    raise KeyError(key_suffix)


def run_seed_sweep(lab, features, label, n_seeds=N_SEEDS, eef_key=EEF_KEY, show_plots=True):
    viz_seeds = TrajectoryVisualizer()
    # Raw normalized EEF action + the initial noise that produced it, for the leak table.
    cols, H = _action_slice(eef_key), len(lab.modality_configs["action"].delta_indices)
    act_norm, noise = [], []
    for seed in range(n_seeds):
        r = lab.denoise(features, seed=seed)
        d = lab.decode_raw_actions(r.action_pred, features.states)
        viz_seeds.add_trajectory(d, f"seed={seed}", eef_key=eef_key)
        act_norm.append(r.action_pred[0, :H, cols].float().cpu().numpy())
        noise.append(r.initial_noise[0, :H, cols].float().cpu().numpy())

    # Stack all trajectories: shape (n_seeds, T+1, 3)
    all_pos = np.stack([t["positions"] for t in viz_seeds.trajectories])
    mean = all_pos.mean(axis=0)   # (T+1, 3)
    var = all_pos.var(axis=0)     # (T+1, 3)
    rng = all_pos.max(axis=0) - all_pos.min(axis=0)  # (T+1, 3)

    comp = ["x", "y", "z"]
    print("Across all timesteps:")
    for i, l in enumerate(comp):
        print(f"  {l}: mean={mean[:, i].mean():.4f}, \tvar={var[:, i].mean():.4f}, \tstd={np.sqrt(var[:, i]).mean():.4f}, \trange={rng[:, i].mean():.4f}")
    print("Across final timestep only:")
    for i, l in enumerate(comp):
        print(f"  {l}: mean={mean[-1, i]:.4f}, \tvar={var[-1, i]:.4f}, \tstd={np.sqrt(var[-1, i]):.4f}, \trange={rng[-1, i]:.4f}")

    # Jerk = 3rd difference of the EEF path = 2nd difference of the per-step deltas.
    # Every line is across seeds of one number per seed (see the markdown above).
    jerk = np.diff(all_pos, n=3, axis=1)                                     # (n_seeds, T-2, 3)
    jabs = np.abs(jerk)
    path_len = np.linalg.norm(np.diff(all_pos, axis=1), axis=2).sum(axis=1)  # (n_seeds,)
    denom = np.maximum(path_len, 1e-9)
    norm_jerk = jabs.sum(axis=1) / denom[:, None]                            # (n_seeds, 3)
    pathjerk = np.linalg.norm(jerk, axis=2).sum(axis=1) / denom              # (n_seeds,)
    print("Jerk |3rd diff of EEF path|, per-seed mean over all timesteps:")
    for i, l in enumerate(comp):
        _seed_stats_line(jabs[:, :, i].mean(axis=1), l)
    print("Jerk |3rd diff of EEF path|, final timestep only:")
    for i, l in enumerate(comp):
        _seed_stats_line(jabs[:, -1, i], l)
    print("Normalized jerk = sum|jerk| / path length, per seed (scale-free; 3D = pathjerk):")
    for i, l in enumerate(comp):
        _seed_stats_line(norm_jerk[:, i], l)
    _seed_stats_line(pathjerk, "3D")

    if show_plots:
        # ONE composite figure per adapter: 3D (left, full height) + 2x3 grid (right).
        comp_up = ["X", "Y", "Z"]
        fig = plt.figure(figsize=(20, 8), constrained_layout=True)
        gs = fig.add_gridspec(2, 5)

        # Left: 3D EEF trajectory — one line per seed, shared start (^) and mean end (square).
        ax3d = fig.add_subplot(gs[:, 0:2], projection="3d")
        for s in range(all_pos.shape[0]):
            p = all_pos[s]
            ax3d.plot(p[:, 0], p[:, 1], p[:, 2], "-", color="C0", linewidth=0.6, alpha=0.4)
        ax3d.scatter(*all_pos[0, 0], marker="^", s=60, color="k", zorder=5)
        ax3d.scatter(*mean[-1], marker="s", s=60, color="r", zorder=5)
        # Equal aspect via bounding box (mirrors TrajectoryVisualizer.plot_eef_3d).
        flat = all_pos.reshape(-1, 3)
        bmin, bmax = flat.min(axis=0), flat.max(axis=0)
        mid = (bmin + bmax) / 2
        half = (bmax - bmin).max() / 2
        if half < 1e-8:
            half = 0.01
        ax3d.set_xlim(mid[0] - half, mid[0] + half)
        ax3d.set_ylim(mid[1] - half, mid[1] + half)
        ax3d.set_zlim(mid[2] - half, mid[2] + half)
        ax3d.set_xlabel("X"); ax3d.set_ylabel("Y"); ax3d.set_zlabel("Z")
        ax3d.set_title(f"3D EEF trajectory ({n_seeds} seeds)", fontsize=10)

        # Top-right: raw X/Y/Z component traces (one line per seed).
        for i in range(3):
            axc = fig.add_subplot(gs[0, 2 + i])
            axc.plot(all_pos[:, :, i].T, color="C0", linewidth=0.5, alpha=0.25)
            axc.set_title(f"EEF {comp_up[i]} ({n_seeds} seeds)", fontsize=9)
            axc.set_xlabel("timestep", fontsize=8)
            axc.grid(True, alpha=0.3)

        # Bottom-right: x/y/z mean +- 1 std spread.
        for i in range(3):
            axs = fig.add_subplot(gs[1, 2 + i])
            std_i = np.sqrt(var[:, i])
            axs.plot(mean[:, i], color="C1", label="mean")
            axs.fill_between(range(mean.shape[0]), mean[:, i] - std_i, mean[:, i] + std_i,
                             alpha=0.3, color="C1", label="±1 std")
            axs.set_title(f"{comp[i]} mean±std", fontsize=9)
            axs.set_xlabel("timestep", fontsize=8)
            axs.grid(True, alpha=0.3)
            if i == 0:
                axs.legend(fontsize=7)

        fig.suptitle(f"Seed spread — {label}", fontsize=13)
        plt.show()

    return {
        "label": label,
        "all_pos": all_pos,  # (n_seeds, T+1, 3) — every plotted seed line
        "mean": mean,
        "var": var,
        "range": rng,
        "final_std": np.sqrt(var[-1]),           # (3,) std at final timestep
        "mean_std": np.sqrt(var).mean(axis=0),   # (3,) std averaged over timesteps
        "jerk_mean": jabs.mean(axis=(0, 1)),     # (3,) mean |jerk| per step
        "pathjerk": float(pathjerk.mean()),      # sum||jerk|| / path length
        "path_len": float(path_len.mean()),
        "act_norm": np.stack(act_norm),          # (n_seeds, H, 3) raw normalized EEF action
        "noise": np.stack(noise),                # (n_seeds, H, 3) its initial noise
    }

In [ ]:
# Cell 4: analyze_all_checkpoints — encode one observation, then run the seed sweep
# for the base model (adapters disabled) and every checkpoint in CHECKPOINTS. Features
# are encoded ONCE (the backbone is LoRA-free); each checkpoint's adapter is loaded into
# the already-injected DiT, fully overwriting the previous checkpoint's A/B matrices.
def analyze_all_checkpoints(obs_path, n_seeds=N_SEEDS, show_obs=True, include_base=INCLUDE_BASE_MODEL):
    obs = DenoisingLab.load_observation(obs_path)
    print("Observation:", obs_path)
    print("Task:", obs["annotation.human.action.task_description"])
    if show_obs:
        DenoisingLab.plot_camera_views(obs, figsize=(14, 4))
        plt.suptitle(f"Camera views — {os.path.basename(obs_path)}", fontsize=12)
        plt.show()

    features = lab.encode_features_from_sim_obs(obs)  # EXPENSIVE — once per obs
    print(f"Backbone features: {features.backbone_features.shape}")

    results, ckpt_of = {}, {}

    # Base model (no LoRA): disable the injected adapters so the DiT produces its
    # pretrained velocity field. Independent of whichever checkpoint is currently
    # loaded, so we run it FIRST — before touching any adapter weights.
    if include_base:
        label = "BASE (no LoRA)"
        print("\n" + "=" * 90)
        print(f"CHECKPOINT: {label}")
        print("=" * 90)
        with disabled_adapters(lab.model.action_head.model):
            results[label] = run_seed_sweep(lab, features, label, n_seeds=n_seeds)
        ckpt_of[label] = ""  # no LoRA

    for label, ckpt_rel in CHECKPOINTS.items():
        ckpt = os.path.join(REPO_ROOT, ckpt_rel)
        print("\n" + "=" * 90)
        print(f"CHECKPOINT: {label}")
        print("=" * 90)
        load_lora_checkpoint(lab.model, ckpt)  # overwrites the previous adapter in place
        results[label] = run_seed_sweep(lab, features, label, n_seeds=n_seeds)
        ckpt_of[label] = ckpt

    # Data behind the composite figures above and the summary bar chart (Cell 6 / Cell 8).
    runs = list(results)
    save_plot_data(
        f"seed_spread__{os.path.splitext(os.path.basename(obs_path))[0]}",
        meta={"obs_path": obs_path, "ckpt_paths": [ckpt_of[k] for k in runs],
              "n_seeds": n_seeds, "eef_key": EEF_KEY,
              "arrays": {"all_pos": "(run, seed, timestep, xyz) EEF path from 0; 3D panel + X/Y/Z traces",
                         "mean/std": "(run, timestep, xyz) across seeds; mean±std panels",
                         "final_std": "(run, xyz) std at the final timestep; summary bar chart",
                         "act_norm/noise": "(run, seed, step, xyz) raw normalized EEF action and its initial noise; leak table"}},
        runs=runs,
        timestep=np.arange(results[runs[0]]["all_pos"].shape[1]),
        all_pos=np.stack([results[k]["all_pos"] for k in runs]),
        mean=np.stack([results[k]["mean"] for k in runs]),
        std=np.sqrt(np.stack([results[k]["var"] for k in runs])),
        final_std=np.stack([results[k]["final_std"] for k in runs]),
        act_norm=np.stack([results[k]["act_norm"] for k in runs]),
        noise=np.stack([results[k]["noise"] for k in runs]),
    )
    return features, results

In [ ]:
# Cell 5: Run the full comparison (base model + all checkpoints) for OBS_PATH.
# OBS_PATH is set to ...ep000_step010.npz in Cell 1. Re-run with a different path for
# any other observation, e.g. analyze_all_checkpoints("/path/to/other.npz").
_features, results = analyze_all_checkpoints(OBS_PATH)

## Cross-checkpoint summary

Final-timestep EEF-position std across the seeds, per checkpoint (and the base model).
Higher std = more action diversity driven by the initial noise; lower std = the
checkpoint has become more "decisive" / peaked for this observation.

In [ ]:
# Cell 6: Cross-checkpoint summary — final-timestep EEF position std across seeds.
labels_short = list(results.keys())
final_std = np.array([results[k]["final_std"] for k in labels_short])  # (n_ckpt, 3)

print(f"{'checkpoint':52s} {'std_x':>8s} {'std_y':>8s} {'std_z':>8s} {'mean':>9s}")
for k, fs in zip(labels_short, final_std):
    print(f"{k:52s} {fs[0]:8.4f} {fs[1]:8.4f} {fs[2]:8.4f} {fs.mean():9.4f}")

x = np.arange(len(labels_short))
w = 0.25
fig, ax = plt.subplots(figsize=(15, 6))
for i, l in enumerate(["x", "y", "z"]):
    ax.bar(x + (i - 1) * w, final_std[:, i], w, label=f"std {l}")
ax.set_xticks(x)
ax.set_xticklabels(labels_short, rotation=70, ha="right", fontsize=7)
ax.set_ylabel("final-timestep EEF std across seeds")
ax.set_title(f"Action diversity by checkpoint — {os.path.basename(OBS_PATH)}")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Re-run the full comparison for a different observation (step 012).
OBS_PATH = "/home/ubuntu/results/npz_save_CoffeeServeMug_v2/ep000_step012.npz"
_features, results = analyze_all_checkpoints(OBS_PATH)

In [ ]:
# Cell 8: Cross-checkpoint summary for the step-012 observation
# (identical to Cell 6, re-run against the results from the cell above).
labels_short = list(results.keys())
final_std = np.array([results[k]["final_std"] for k in labels_short])  # (n_ckpt, 3)

print(f"{'checkpoint':52s} {'std_x':>8s} {'std_y':>8s} {'std_z':>8s} {'mean':>9s}")
for k, fs in zip(labels_short, final_std):
    print(f"{k:52s} {fs[0]:8.4f} {fs[1]:8.4f} {fs[2]:8.4f} {fs.mean():9.4f}")

x = np.arange(len(labels_short))
w = 0.25
fig, ax = plt.subplots(figsize=(15, 6))
for i, l in enumerate(["x", "y", "z"]):
    ax.bar(x + (i - 1) * w, final_std[:, i], w, label=f"std {l}")
ax.set_xticks(x)
ax.set_xticklabels(labels_short, rotation=70, ha="right", fontsize=7)
ax.set_ylabel("final-timestep EEF std across seeds")
ax.set_title(f"Action diversity by checkpoint — {os.path.basename(OBS_PATH)}")
ax.legend()
plt.tight_layout()
plt.show()

## Leak table

Noise seeds are fixed, so seed `s` starts every checkpoint from the same initial noise. Two
independent measures of how much of that noise reaches the actions, both on the step-to-step
part of the motion (rfft bins 3+ along the 16 steps):

- **`abs leak`**: the raw normalized EEF action regressed on the seed's own initial noise (both
  saved by `run_seed_sweep`). It is the fraction of the noise that passes straight through.
  `noise corr` tests that reading: near 1 means the wiggle is the noise passing through step
  for step, near 0 means the action's roughness is something else.
- **`leak gain`**: the decoded EEF deltas regressed on the same seed's pattern from `LEAK_REF`
  at step010 (1.0 = `LEAK_REF`, base ≈ 0.09 in v4); `fp corr` is how closely they match.

If both describe the same leak, `abs/ref` (abs leak relative to `LEAK_REF`'s) should match
`leak gain`. If they agree here, later notebooks can drop `LEAK_REF`. `pathjerk` is
`sum ‖jerk‖ / path length` and `x base` compares it with BASE. Everything is computed from the
saved `seed_spread` files, so this cell needs no GPU and can be re-run offline. It raises if the
initial noise differs across runs, because then no leak measure is valid.


In [ ]:
# Cell 8.5: leak table — how much of each seed's initial noise reaches its actions.
# Reads the seed_spread files written above; no extra forwards (see the markdown).
def _hp(d):
    """Step-to-step part of (n_seeds, T, 3) arrays: remove the seed mean, drop rfft bins 0-2."""
    f = np.fft.rfft(d - d.mean(axis=0), axis=1)
    f[:, :3] = 0
    return np.fft.irfft(f, n=d.shape[1], axis=1)


def _seed_corr(A, B):
    """Mean over seeds of the correlation between A[s] and B[s]."""
    return float(np.mean([np.corrcoef(a, b)[0, 1]
                          for a, b in zip(A.reshape(len(A), -1), B.reshape(len(B), -1))]))


def leak_table(obs_tags=("ep000_step010", "ep000_step012"), ref=LEAK_REF):
    load = lambda tag: np.load(os.path.join(PLOT_DATA_DIR, f"seed_spread__{tag}.npz"))
    z = load(obs_tags[0])
    runs = [str(r) for r in z["runs"]]
    F = _hp(np.diff(z["all_pos"][runs.index(ref)], axis=1))       # reference fingerprint
    out = {}
    for tag in obs_tags:
        z = load(tag)
        runs = [str(r) for r in z["runs"]]
        if not all(np.array_equal(z["noise"][r], z["noise"][0]) for r in range(len(runs))):
            raise RuntimeError(f"{tag}: initial noise differs across runs, so seeds are not "
                               f"shared and no leak measure is valid")
        he = _hp(z["noise"][0])
        k_abs = [float((_hp(z["act_norm"][r]) * he).sum() / (he * he).sum()) for r in range(len(runs))]
        k_ref = k_abs[runs.index(ref)]
        print(f"\n{tag}")
        print(f"  {'run':24s} {'pathjerk':>8s} {'x base':>6s} {'|jerk|/step':>11s} {'path len':>8s} "
              f"{'end std':>7s} {'leak gain':>9s} {'fp corr':>7s} {'abs leak':>8s} {'abs/ref':>7s} "
              f"{'noise corr':>10s}")
        base_pj = None
        for r, run in enumerate(runs):
            pos = z["all_pos"][r]
            D, j = np.diff(pos, axis=1), np.diff(pos, n=3, axis=1)
            L = np.linalg.norm(D, axis=2).sum(axis=1)
            pj = float((np.linalg.norm(j, axis=2).sum(axis=1) / np.maximum(L, 1e-9)).mean())
            base_pj = pj if base_pj is None else base_pj         # first run is BASE
            h = _hp(D)
            gain = float((h * F).sum() / (F * F).sum())
            ncorr = _seed_corr(_hp(z["act_norm"][r]), he)
            out[(tag, run)] = {"pathjerk": pj, "leak_gain": gain, "fp_corr": _seed_corr(h, F),
                               "abs_leak": k_abs[r], "noise_corr": ncorr}
            print(f"  {run:24s} {pj:8.3f} {pj / base_pj:5.1f}x {np.abs(j).mean():11.4f} {L.mean():8.2f} "
                  f"{pos[:, -1].std(axis=0).mean():7.3f} {gain:9.3f} {out[(tag, run)]['fp_corr']:7.2f} "
                  f"{k_abs[r]:8.4f} {k_abs[r] / k_ref:7.3f} {ncorr:10.2f}")
    return out


leak = leak_table()


## Solver convergence

Path jerk, relative dispersion and mean-trajectory drift against the number of Euler steps
(4 / 8 / 16 / 32) on a shared 25-seed set. In v4, base-like fields got rougher with more steps
(the base is tuned for 4), while the old-target checkpoints stayed flat around 0.5–0.6.


In [ ]:
# Cell 13: solver_convergence — sweep num_steps on a shared seed set.
# Cost: n_seeds * sum(steps_list) forwards per checkpoint (25 * 60 = 1500 at the
# defaults). Nothing here runs the simulator or produces a success rate.
def _traj_from_chunk(lab, features, a, eef_key=EEF_KEY):
    """Decode a raw chunk to the integrated EEF path, matching
    TrajectoryVisualizer.add_trajectory (0 prepended + cumsum)."""
    d = lab.decode_raw_actions(a, features.states)
    deltas = d[eef_key][0][:, :3]
    return np.concatenate([np.zeros((1, 3)), np.cumsum(deltas, axis=0)], axis=0), deltas


def solver_convergence(lab, features, label, steps_list=(4, 8, 16, 32), n_seeds=25):
    out = {"label": label, "steps": list(steps_list)}
    per_steps = {}
    for ns in steps_list:
        paths, jerks = [], []
        for s in range(n_seeds):                       # SAME seeds for every ns
            res = lab.denoise(features, seed=s, num_steps=ns)
            pos, deltas = _traj_from_chunk(lab, features, res.action_pred)
            paths.append(pos)
            L = float(np.linalg.norm(deltas, axis=1).sum())
            J = float(np.linalg.norm(np.diff(pos, n=3, axis=0), axis=1).sum())
            jerks.append(J / max(L, 1e-9))
        P = np.stack(paths)                            # (n_seeds, T+1, 3)
        mean_traj = P.mean(axis=0)
        disp = float(np.linalg.norm(mean_traj[-1]))
        per_steps[ns] = {
            "pathjerk": float(np.mean(jerks)),
            "rel_disp": float(np.sqrt(P[:, -1].var(axis=0)).mean() / max(disp, 1e-9)),
            "mean_traj": mean_traj,
            "disp": disp,
        }
    ref = per_steps[steps_list[0]]
    for ns in steps_list:                              # drift vs the 4-step mean
        per_steps[ns]["drift"] = float(
            np.linalg.norm(per_steps[ns]["mean_traj"] - ref["mean_traj"], axis=1).mean()
            / max(ref["disp"], 1e-9))
    out["per_steps"] = per_steps

    print(f"  {'num_steps':>9s} {'pathjerk':>9s} {'vs 4step':>9s} "
          f"{'rel_disp':>9s} {'vs 4step':>9s} {'drift':>8s}")
    j4, d4 = ref["pathjerk"], ref["rel_disp"]
    for ns in steps_list:
        v = per_steps[ns]
        print(f"  {ns:9d} {v['pathjerk']:9.5f} {v['pathjerk'] / max(j4, 1e-12):9.2f} "
              f"{v['rel_disp']:9.4f} {v['rel_disp'] / max(d4, 1e-12):9.2f} "
              f"{v['drift']:8.4f}")
    return out


def run_solver_sweep(obs_path, steps_list=(4, 8, 16, 32), n_seeds=25,
                     include_base=True):
    obs = DenoisingLab.load_observation(obs_path)
    features = lab.encode_features_from_sim_obs(obs)
    print(f"Observation: {obs_path}   ({n_seeds} seeds, shared across num_steps)\n")
    results, ckpt_of = {}, {}
    if include_base:
        print("=" * 90); print("BASE (no LoRA)"); print("=" * 90)
        with disabled_adapters(lab.model.action_head.model):
            results["BASE"] = solver_convergence(lab, features, "BASE",
                                                 steps_list, n_seeds)
        ckpt_of["BASE"] = ""
    for lbl, ckpt_rel in CHECKPOINTS.items():
        print("\n" + "=" * 90); print(f"CHECKPOINT: {lbl}"); print("=" * 90)
        load_lora_checkpoint(lab.model, os.path.join(REPO_ROOT, ckpt_rel))
        results[lbl] = solver_convergence(lab, features, lbl, steps_list, n_seeds)
        ckpt_of[lbl] = os.path.join(REPO_ROOT, ckpt_rel)

    # Data behind the 3-panel figure below.
    runs, sl = list(results), list(steps_list)

    def _per(f):
        return np.array([[results[k]["per_steps"][n][f] for n in sl] for k in runs])

    save_plot_data(
        f"solver_convergence__{os.path.splitext(os.path.basename(obs_path))[0]}",
        meta={"obs_path": obs_path, "ckpt_paths": [ckpt_of[k] for k in runs],
              "n_seeds": n_seeds,
              "arrays": {"pathjerk/rel_disp/drift": "(run, num_steps); the three panels",
                         "disp": "(run, num_steps) ||final point of the seed-mean path||",
                         "mean_traj": "(run, num_steps, timestep, xyz) seed-mean EEF path"}},
        runs=runs, steps=np.array(sl),
        pathjerk=_per("pathjerk"), rel_disp=_per("rel_disp"), drift=_per("drift"),
        disp=_per("disp"), mean_traj=_per("mean_traj"),
    )

    return results


_solver_010 = run_solver_sweep(
    "/home/ubuntu/results/npz_save_CoffeeServeMug_v2/ep000_step010.npz")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), constrained_layout=True)
for k, v in _solver_010.items():
    sl = v["steps"]
    axes[0].plot(sl, [v["per_steps"][n]["pathjerk"] for n in sl], "-o", label=k)
    axes[1].plot(sl, [v["per_steps"][n]["rel_disp"] for n in sl], "-o", label=k)
    axes[2].plot(sl, [v["per_steps"][n]["drift"] for n in sl], "-o", label=k)
for ax, ttl, yl in zip(axes,
        ["normalized path jerk vs num_steps (plateau => solver-limited)",
         "relative dispersion vs num_steps",
         "mean-trajectory drift vs 4-step (the cost of changing num_steps)"],
        ["jerk / path length", "std / ||displacement||", "relative drift"]):
    ax.set_title(ttl, fontsize=9); ax.set_xlabel("num Euler steps"); ax.set_ylabel(yl)
    ax.set_xscale("log", base=2); ax.grid(True, alpha=0.3); ax.legend(fontsize=7)
plt.show()


## Residual roughness

Per τ-center: the velocity residual `r = v − (a − ε)` along the FM path (`M(r)` is its size,
`HF(r)` its roughness along the horizon, 0 = smooth and 1 = white), the generated chunk's own
roughness (`act_HF`), the implied endpoint's roughness (`HF(a_hat)`), path jerk and the residual
spectrum. One table per observation; the figure after step012 plots both.


In [ ]:
# Cell 9: Action-dim layout, read from the CHECKPOINT (self-verifying).
#
# Mirrors processing_gr00t_n1d6.decode_action (:239-246) and
# grpo_server.compute_action_mask (:278): walk this embodiment's action
# modality_keys IN ORDER, take each key's normalized `dim`, accumulate contiguous
# slices. If this checkpoint's layout differs from the expected PandaOmron one,
# you see it in the printout rather than silently masking the wrong columns.
TAG = lab.embodiment_tag.value
_acfg = lab.modality_configs["action"]
_norm = lab.processor.state_action_processor.norm_params[TAG]["action"]

ACTION_HORIZON_VALID = len(_acfg.delta_indices)          # 16 for PandaOmron
_layout, _start = [], 0
for _key in _acfg.modality_keys:
    _d = _norm[_key]["dim"]
    _d = int(_d.item() if hasattr(_d, "item") else _d)
    _layout.append((_key, _start, _start + _d))
    _start += _d
N_VALID_DIMS = _start

# Discrete 0/1 channels — EXCLUDED from every roughness measure. robocasa's
# PandaOmronKeyConverter.unmap_action thresholds both at 0.5, so a grasp and a
# base-mode switch ARE step functions and must stay free to step.
DISCRETE_KEYS = {"gripper_close", "control_mode"}

# base_motion is gated by control_mode (HybridMobileBase makes arm and base
# mutually exclusive), so its roughness can be inert in arm mode. Keep it OUT of
# the default set so the constraint's capacity goes to the channel the plots
# actually show; flip to True to include it and compare.
INCLUDE_BASE_MOTION = False

C_DIMS, C_KEYS = [], []
for _key, _lo, _hi in _layout:
    _short = _key.split(".")[-1]
    if _short in DISCRETE_KEYS:
        continue
    if _short == "base_motion" and not INCLUDE_BASE_MOTION:
        continue
    C_DIMS.extend(range(_lo, _hi))
    C_KEYS.append(_short)

# Measure the FULL valid horizon, not just the executed prefix. Three reasons:
#   1. compute_fm_log_prob masks with action_mask = the full valid rectangle
#      (16x12 here), so ref_mse / the ratio / every advantage already cover all 16
#      steps. Measuring M over 8 would make smooth/M_mean non-comparable to
#      ref_mse/mean.
#   2. 14 second-differences instead of 6 -> less than half the per-row variance,
#      which is what keeps a per-row relu gate from chattering.
#   3. n_action_steps is a DEPLOYMENT knob; the 16-step horizon is a property of
#      the checkpoint (delta_indices). Baking the former into the objective means a
#      later --n_action_steps 16 run executes steps never trained to be smooth.
# The DiT also emits the chunk jointly (self-attention over action tokens), so
# nothing keeps tail degradation out of the head.
H_MEASURE = ACTION_HORIZON_VALID

# Executed prefix — used ONLY to split the diagnostic into head / tail below, so
# we can see empirically whether roughness is uniform in h or tail-concentrated.
N_ACTION_STEPS = 8

# tau_centers: GRPOConfig default (late-biased). Keep in sync with the run.
TAU_CENTERS = [0.0, 0.25, 0.35, 0.5, 0.6, 0.75]

print(f"embodiment={TAG}  valid_horizon={ACTION_HORIZON_VALID}  valid_dims={N_VALID_DIMS}")
print(f"raw model output = ({lab.action_horizon}, {lab.action_dim}) -> valid rectangle "
      f"[:{ACTION_HORIZON_VALID}, :{N_VALID_DIMS}]")
print("\naction dim layout (order comes from the checkpoint, NOT from the repo):")
for _key, _lo, _hi in _layout:
    _short = _key.split(".")[-1]
    _tag = "DISCRETE - excluded" if _short in DISCRETE_KEYS else (
        "excluded (INCLUDE_BASE_MOTION=False)"
        if _short == "base_motion" and not INCLUDE_BASE_MOTION else "in C")
    print(f"  [{_lo:2d}:{_hi:2d}]  {_short:28s} dim={_hi - _lo}  {_tag}")
print(f"\nC = {C_KEYS} -> {len(C_DIMS)} dims: {C_DIMS}")
print(f"h range measured: 0..{H_MEASURE - 1} (FULL valid horizon), "
      f"{max(0, H_MEASURE - 2)} second-differences")
print(f"diagnostic split: head = h<{N_ACTION_STEPS} (executed), "
      f"tail = h>={N_ACTION_STEPS} (discarded at n_action_steps={N_ACTION_STEPS})")
print(f"tau centers: {TAU_CENTERS}")


In [ ]:
# Cell 10: measure_roughness — residual spectrum (M, HF) + scale-controlled path jerk.
#
# One pass over seeds computes BOTH families, so this adds no denoise() calls
# beyond the ones it needs:
#   * residual: for each tau, build the FM interpolant x_tau = (1-tau)*eps + tau*a,
#     forward the DiT once, and take r = v_theta - (a - eps). This is exactly the
#     tensor compute_fm_log_prob squares at fm_log_prob.py:253.
#   * path jerk: decode the chunk to EEF deltas, integrate to positions the same
#     way TrajectoryVisualizer.add_trajectory does (0 prepended + cumsum), then
#     report sum(|3rd difference|) / path_length. The NORMALIZATION is the point:
#     LoRA paths are 28-43% longer, so RAW jerk rises mechanically and cannot
#     distinguish "rougher" from "bigger".
#
# Slicing to the valid rectangle happens BEFORE differencing — differencing the
# padded (50, 128) output would straddle the pad boundary and return garbage.
#
# Not batched over seeds on purpose: features has batch 1, and broadcasting a
# batch-1 vl_embeds against batch-S actions in cross-attention is a silent-
# wrongness risk. n_seeds=25 is plenty for a distributional statistic.
import numpy as np
import torch


def _d2(u):
    """Second difference along the horizon axis (axis=1) of an (N, H, D) array."""
    return u[:, 2:] - 2.0 * u[:, 1:-1] + u[:, :-2]


def measure_roughness(lab, features, label, n_seeds=25, tau_centers=None,
                      dims=None, h_measure=None, num_steps=4, eef_key=EEF_KEY):
    tau_centers = TAU_CENTERS if tau_centers is None else tau_centers
    dims = C_DIMS if dims is None else dims
    H = H_MEASURE if h_measure is None else h_measure
    nb = lab.num_timestep_buckets

    per_tau = {tau: {"M": [], "R": [], "M_head": [], "R_head": [],
                     "M_tail": [], "R_tail": [], "M_ahat": [], "R_ahat": []}
               for tau in tau_centers}
    act_M, act_R, path_jerk, path_len = [], [], [], []

    # Head/tail windows for the diagnostic split. _d2 returns (N, H-2, D), so
    # index j covers h in [j, j+2]. "head" = windows entirely inside the executed
    # prefix; "tail" = windows entirely inside the discarded remainder. The two
    # straddling windows belong to neither and are dropped from the split only
    # (never from the full measure).
    he = min(N_ACTION_STEPS, H)
    head_sl, tail_sl = slice(0, max(he - 2, 0)), slice(he, max(H - 2, 0))

    # Full power spectrum of r along h. HF is only a 2-moment SUMMARY of this, and
    # it SATURATES: a residual that is even 50% white already reads HF~0.93, so if
    # the base residual is unstructured error (the null hypothesis for a well-fit
    # regression) HF_base ~ 1 and there is <2.7x headroom before the 8/3 ceiling.
    # The spectrum says directly whether HF is a usable instrument here: flat =
    # white = HF pinned at 1 = use the M cap instead, not the HF hinge.
    spec_acc = np.zeros(H // 2 + 1)
    spec_n = 0

    for s in range(n_seeds):
        res = lab.denoise(features, seed=s, num_steps=num_steps)
        a, eps = res.action_pred, res.initial_noise
        v_target = a - eps                                    # FM velocity target

        # --- roughness of the produced chunk itself (normalized action space) ---
        a_v = a[:, :H, :].float()[:, :, dims]
        act_M.append(a_v.pow(2).mean().item())
        act_R.append(_d2(a_v).pow(2).mean().item())

        # --- scale-controlled jerk of the decoded EEF path (physical units) ---
        d = lab.decode_raw_actions(a, features.states)
        deltas = d[eef_key][0][:, :3]                          # (T, 3)
        pos = np.concatenate([np.zeros((1, 3)), np.cumsum(deltas, axis=0)], axis=0)
        L = float(np.linalg.norm(deltas, axis=1).sum())
        J = float(np.linalg.norm(np.diff(pos, n=3, axis=0), axis=1).sum())
        path_len.append(L)
        path_jerk.append(J / max(L, 1e-9))

        # --- velocity residual at each tau center ---
        for tau in tau_centers:
            x_tau = (1.0 - tau) * eps + tau * a
            # denoise_single_step computes t_cont = step_idx / num_total_steps, so
            # passing num_total_steps = num_timestep_buckets hits tau exactly.
            v, _ = lab.denoise_single_step(features, x_tau, round(tau * nb), nb)
            r = (v - v_target)[:, :H, :].float()[:, :, dims]
            d2r = _d2(r)
            # The model's IMPLIED ENDPOINT, a_hat = x_tau + (1-tau)*v = a + (1-tau)*r.
            # At tau=0, x_tau IS eps, so this evaluation sits ON the sampler's path and
            # a_hat is literally the 1-step Euler endpoint. This is the differentiable
            # quantity CLOSEST to the generated chunk — which matters because the
            # chunk's spectrum (act_HF) degrades far more than the residual's.
            ahat = (x_tau + (1.0 - tau) * v)[:, :H, :].float()[:, :, dims]
            per_tau[tau]["M_ahat"].append(ahat.pow(2).mean().item())
            per_tau[tau]["R_ahat"].append(_d2(ahat).pow(2).mean().item())
            per_tau[tau]["M"].append(r.pow(2).mean().item())
            per_tau[tau]["R"].append(d2r.pow(2).mean().item())
            # pooled over tau: normalized |rfft|^2 along h, averaged over dims
            _p = np.abs(np.fft.rfft(r.cpu().numpy(), axis=1)) ** 2
            spec_acc += (_p / max(_p.sum(axis=1, keepdims=True).mean(), 1e-30)
                         ).mean(axis=(0, 2))
            spec_n += 1
            # head / tail split — diagnostic only, never used by the loss
            if head_sl.stop > head_sl.start:
                per_tau[tau]["M_head"].append(r[:, :he].pow(2).mean().item())
                per_tau[tau]["R_head"].append(d2r[:, head_sl].pow(2).mean().item())
            if tail_sl.stop > tail_sl.start:
                per_tau[tau]["M_tail"].append(r[:, he:].pow(2).mean().item())
                per_tau[tau]["R_tail"].append(d2r[:, tail_sl].pow(2).mean().item())

    def agg(key, tau):
        vals = per_tau[tau][key]
        return float(np.mean(vals)) if vals else float("nan")

    taus = list(tau_centers)
    M = np.array([agg("M", t) for t in taus])
    R = np.array([agg("R", t) for t in taus])
    HF = R / (6.0 * np.maximum(M, 1e-12))          # 0 = smooth, 1 = white in h
    HF_head = (np.array([agg("R_head", t) for t in taus])
               / (6.0 * np.maximum([agg("M_head", t) for t in taus], 1e-12)))
    HF_tail = (np.array([agg("R_tail", t) for t in taus])
               / (6.0 * np.maximum([agg("M_tail", t) for t in taus], 1e-12)))
    HF_ahat = (np.array([agg("R_ahat", t) for t in taus])
               / (6.0 * np.maximum([agg("M_ahat", t) for t in taus], 1e-12)))

    out = {
        "label": label, "taus": taus,
        "M": M, "R": R, "HF": HF,
        "HF_head": HF_head, "HF_tail": HF_tail, "HF_ahat": HF_ahat,
        "HF_ahat_mean": float(np.nanmean(HF_ahat)),
        "M_a": float(np.mean(act_M)),
        "M_mean": float(M.mean()), "HF_mean": float(HF.mean()),
        "HF_head_mean": float(np.nanmean(HF_head)),
        "HF_tail_mean": float(np.nanmean(HF_tail)),
        # the chunk's own spectrum, same normalization
        "act_HF": float(np.mean(act_R) / max(np.mean(act_M), 1e-12) / 6.0),
        "spectrum": spec_acc / max(spec_n, 1),      # length H//2+1, sums to ~1
        "path_jerk": float(np.mean(path_jerk)),
        "path_jerk_p90": float(np.percentile(path_jerk, 90)),
        "path_len": float(np.mean(path_len)),
        "n_seeds": n_seeds,
    }
    print(f"  M(r) mean over tau = {out['M_mean']:.5f}   "
          f"(compare ref_mse/mean in TB)")
    print(f"  HF(r) mean over tau = {out['HF_mean']:.4f}   "
          f"[0=smooth, 1=white]")
    print(f"  HF per tau         = " + "  ".join(f"{t:.2f}:{h:.3f}"
                                                 for t, h in zip(taus, HF)))
    print(f"  HF head/tail       = {out['HF_head_mean']:.4f} (h<{he}) / "
          f"{out['HF_tail_mean']:.4f} (h>={he})   "
          f"[tail/head = {out['HF_tail_mean'] / max(out['HF_head_mean'], 1e-9):.2f}x]")
    print(f"  HF(action chunk)   = {out['act_HF']:.4f}   M(a) = {out['M_a']:.4f}"
          f"   [M(a)/M(r) = {out['M_a'] / max(out['M_mean'], 1e-12):.3g}x]")
    print(f"  HF(a_hat) per tau  = " + "  ".join(
        f"{t:.2f}:{h:.4f}" for t, h in zip(taus, HF_ahat))
          + f"   mean {out['HF_ahat_mean']:.4f}")
    print(f"  path jerk / length = {out['path_jerk']:.5f}  "
          f"(p90 {out['path_jerk_p90']:.5f}), mean path length {out['path_len']:.3f}")
    sp = out["spectrum"]
    print(f"  |rfft(r)|^2 by bin  = " + " ".join(f"{p:.3f}" for p in sp))
    print(f"    flat ({1.0 / len(sp):.3f} each) = white = HF pinned near 1;"
          f"  hi/lo half = {sp[len(sp) // 2:].sum() / max(sp[:len(sp) // 2].sum(), 1e-12):.2f}")
    return out


In [ ]:
# Cell 11: run the measurement for the base model + every checkpoint, then print
# the roughness table. Mirrors analyze_all_checkpoints: encode features ONCE (the
# backbone is LoRA-free), run base with the adapters disabled, then load each
# checkpoint in place.
HF_N_SEEDS = 25          # 25 x (4 denoise + 6 probe) forwards per checkpoint


def measure_all_checkpoints(obs_path, n_seeds=HF_N_SEEDS, include_base=True):
    obs = DenoisingLab.load_observation(obs_path)
    features = lab.encode_features_from_sim_obs(obs)
    print(f"Observation: {obs_path}")
    print(f"C = {C_KEYS} ({len(C_DIMS)} dims), h 0..{H_MEASURE - 1}, "
          f"{n_seeds} seeds\n")

    results, ckpt_of = {}, {}
    if include_base:
        print("=" * 90); print("BASE (no LoRA)"); print("=" * 90)
        with disabled_adapters(lab.model.action_head.model):
            results["BASE"] = measure_roughness(lab, features, "BASE",
                                                n_seeds=n_seeds)
        ckpt_of["BASE"] = ""
    for label, ckpt_rel in CHECKPOINTS.items():
        print("\n" + "=" * 90); print(f"CHECKPOINT: {label}"); print("=" * 90)
        load_lora_checkpoint(lab.model, os.path.join(REPO_ROOT, ckpt_rel))
        results[label] = measure_roughness(lab, features, label, n_seeds=n_seeds)
        ckpt_of[label] = os.path.join(REPO_ROOT, ckpt_rel)

    # Data behind this observation's row of the Cell 12 figure (HF / M vs tau, roughness
    # bars, spectrum) plus the other per-run measurements printed above.
    runs = list(results)
    keys = ("M", "R", "HF", "HF_head", "HF_tail", "HF_ahat", "spectrum",
            "M_a", "act_HF", "path_jerk", "path_jerk_p90", "path_len")
    save_plot_data(
        f"roughness__{os.path.splitext(os.path.basename(obs_path))[0]}",
        meta={"obs_path": obs_path, "ckpt_paths": [ckpt_of[k] for k in runs],
              "n_seeds": n_seeds, "C_KEYS": C_KEYS, "h_measure": H_MEASURE,
              "arrays": {"M/R/HF/HF_head/HF_tail/HF_ahat": "(run, tau)",
                         "spectrum": "(run, freq_bin) |rfft(r)|^2 along h",
                         "M_a/act_HF/path_jerk/path_jerk_p90/path_len": "(run,)"}},
        runs=runs, taus=np.array(results[runs[0]]["taus"]),
        **{k: np.stack([np.asarray(results[r][k]) for r in runs]) for k in keys},
    )
    return features, results


def report(results, obs_name=""):
    base = results.get("BASE")
    print("\n" + "=" * 90)
    print(f"ROUGHNESS TABLE — {obs_name}")
    print("=" * 90)
    print(f"{'run':24s} {'M(r)':>9s} {'M ratio':>8s} {'HF(r)':>7s} "
          f"{'HF ratio':>9s} {'HFahat':>8s} {'ahat r':>7s} "
          f"{'act_HF':>8s} {'aHF r':>7s} {'pathjerk':>9s} {'pj ratio':>9s}")
    for k, v in results.items():
        mr = v["M_mean"] / base["M_mean"] if base else float("nan")
        hr = v["HF_mean"] / base["HF_mean"] if base else float("nan")
        pr = v["path_jerk"] / base["path_jerk"] if base else float("nan")
        ar = v["HF_ahat_mean"] / base["HF_ahat_mean"] if base else float("nan")
        cr = v["act_HF"] / max(base["act_HF"], 1e-12) if base else float("nan")
        print(f"{k:24s} {v['M_mean']:9.5f} {mr:8.2f} {v['HF_mean']:7.4f} "
              f"{hr:9.2f} {v['HF_ahat_mean']:8.4f} {ar:7.2f} "
              f"{v['act_HF']:8.4f} {cr:7.1f} {v['path_jerk']:9.5f} {pr:9.2f}")



_feat_010, res_010 = measure_all_checkpoints(
    "/home/ubuntu/results/npz_save_CoffeeServeMug_v2/ep000_step010.npz")
report(res_010, "ep000_step010.npz")


In [ ]:
# Cell 12: same measurement for step012, then plot HF(tau) and M(tau) for both
# observations. HF is the new signal; M is the sanity check against ref_mse/mean.
_feat_012, res_012 = measure_all_checkpoints(
    "/home/ubuntu/results/npz_save_CoffeeServeMug_v2/ep000_step012.npz")
report(res_012, "ep000_step012.npz")

fig, axes = plt.subplots(2, 4, figsize=(22, 8), constrained_layout=True)
for row, (res, name) in enumerate([(res_010, "step010"), (res_012, "step012")]):
    for k, v in res.items():
        axes[row, 0].plot(v["taus"], v["HF"], "-o", label=k, markersize=4)
        axes[row, 1].plot(v["taus"], v["M"], "-o", label=k, markersize=4)
    axes[row, 0].set_title(f"HF(r) vs tau — {name}   [0=smooth, 1=white]", fontsize=10)
    axes[row, 0].set_xlabel("tau"); axes[row, 0].set_ylabel("HF")
    axes[row, 0].axhline(1.0, color="k", ls=":", lw=0.8)
    axes[row, 1].set_title(f"M(r) = mean(r^2) vs tau — {name}", fontsize=10)
    axes[row, 1].set_xlabel("tau"); axes[row, 1].set_yscale("log")
    labels = list(res.keys())
    x = np.arange(len(labels))
    axes[row, 2].bar(x - 0.2, [res[k]["path_jerk"] for k in labels], 0.4,
                     label="jerk / path length")
    axes[row, 2].bar(x + 0.2, [res[k]["act_HF"] for k in labels], 0.4,
                     label="HF(action chunk)")
    axes[row, 2].set_xticks(x)
    axes[row, 2].set_xticklabels([l.replace("iter_", "it") for l in labels],
                                 rotation=30, ha="right", fontsize=8)
    axes[row, 2].set_title(f"scale-controlled roughness — {name}", fontsize=10)
    for k, v in res.items():
        sp = v["spectrum"]
        axes[row, 3].plot(np.arange(len(sp)), sp, "-o", label=k, markersize=4)
    axes[row, 3].axhline(1.0 / len(res["BASE"]["spectrum"]), color="k", ls=":", lw=0.8,
                         label="flat = white")
    axes[row, 3].set_title(f"|rfft(r)|^2 along h — {name}", fontsize=10)
    axes[row, 3].set_xlabel("frequency bin"); axes[row, 3].set_yscale("log")
    for c in range(4):
        axes[row, c].grid(True, alpha=0.3)
        axes[row, c].legend(fontsize=7)
plt.show()
